# S1 — Feature Engineering Pipeline

**Course**: Econometrics A.Y. 2025/26 — Politecnico di Milano  
**Author**: Alessio Porrini

---

## Purpose

This notebook documents the **complete feature engineering pipeline** that transforms
raw SSVI calibration outputs into the final modelling dataset used in:

- **Notebook B** — SSVI parameter dynamics (ARMA/ARMAX, VAR, PCA)
- **Notebook C** — Realized volatility forecasting (HAR-RV, SSVI-enhanced models)

All feature constructions are **look-ahead free**: every quantity computed at time $t$
uses only information available at or before $t$.

## Data sources

| Source | Content | Access |
|--------|---------|--------|
| GitHub (Data/) | SSVI parameters, no-arbitrage diagnostics | `pandas.read_csv` |
| Yahoo Finance (`^GSPC`) | S&P 500 daily prices for RV | `yfinance` |
| FRED (`VIXCLS`) | VIX index (ARMAX exogenous, stress indicator) | `fredapi` (personal API key required) |

## Feature families

| Family | Features | Used in |
|--------|----------|--------|
| SSVI levels | α, β, ρ, η, γ | B, C |
| SSVI first differences | Δα, Δβ, Δρ, Δη, Δγ | B |
| SSVI-derived | ATM IV, max\_cond1, skew\_stress | B, C |
| HAR-RV components | RV_d, RV_w, RV_m | C |
| VIX (exogenous only) | log-VIX, Δlog-VIX, VIX z-score | B (ARMAX benchmark) |

> **Portability principle**: all primary forecasting models in Notebook C use
> only SSVI parameters as inputs — no VIX, no market micro-structure data.
> This makes the models applicable to any liquid options market where SSVI
> calibration is available.

## 1. Imports and configuration

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

try:
    from fredapi import Fred
    HAS_FREDAPI = True
except ImportError:
    HAS_FREDAPI = False
    print('fredapi not available — VIX market data will be skipped (pip install fredapi)')

# !!! REQUIRED -- INSERT YOUR PERSONAL FRED API KEY HERE (used to fetch VIX below) !!!
# EVERY USER MUST REPLACE THIS WITH THEIR OWN FREE KEY: https://fred.stlouisfed.org/docs/api/api_key.html
FRED_API_KEY = "PASTE_YOUR_FRED_API_KEY_HERE"   # <-- INSERT YOUR PERSONAL FRED API KEY

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

# ── Data sources ──────────────────────────────────────────────────────────────
GITHUB = 'https://raw.githubusercontent.com/aporrini/Econometrics-Volatility-Surface-Dynamics/main/Data'
SSVI_URL = f'{GITHUB}/ssvi_all_dates_clean_results.csv'
NA_URL   = f'{GITHUB}/no_arbitrage_clean_results.csv'

# Market data window matching the SSVI dataset (2010-01-01 to 2020-12-31)
MKT_START, MKT_END = '2009-12-01', '2021-01-31'

print('Configuration ready.')
print(f'  SSVI URL : {SSVI_URL}')
print(f'  NA   URL : {NA_URL}')
print(f'  fredapi available: {HAS_FREDAPI}')

## 2. Load SSVI calibration results

The SSVI surface is parametrized as (Gatheral & Jacquier 2014):

$$
\omega(k, T) = \frac{\theta_T}{2}\left\{1 + \rho\phi k
  + \sqrt{(\phi k + \rho)^2 + 1 - \rho^2}\right\}
$$

where $\theta_T = e^\alpha T^\beta$ is the ATM total variance and
$\phi = \eta\,\theta_T^{-\gamma}/(1+\eta\,\theta_T^{1-\gamma})$ governs curvature.

**Parameters estimated per day**: $(\alpha, \beta, \rho, \eta, \gamma)$.

In [ ]:
# Load SSVI parameters
try:
    ssvi = pd.read_csv(SSVI_URL)
    print(f'SSVI loaded from GitHub:   {len(ssvi):,} rows')
except Exception as e:
    print(f'GitHub load failed ({e}). Trying local fallback...')
    import os
    LOCAL = os.path.join('..', 'Data')
    ssvi = pd.read_csv(os.path.join(LOCAL, 'ssvi_all_dates_clean_results.csv'))
    print(f'SSVI loaded locally:       {len(ssvi):,} rows')

# Filter successful calibrations
if 'success' in ssvi.columns:
    ssvi = ssvi[ssvi['success']].copy()

# Ensure time_elapsed is integer index
ssvi['time_elapsed'] = ssvi['time_elapsed'].astype(int)
ssvi = ssvi.sort_values('time_elapsed').reset_index(drop=True)

PARAM_COLS = ['alpha', 'beta', 'rho', 'eta', 'gamma']
print(f'  Successful calibrations: {len(ssvi):,}')
print(f'  Time range: {ssvi["time_elapsed"].min()} – {ssvi["time_elapsed"].max()}')
ssvi[PARAM_COLS].describe().round(4)

In [ ]:
# Load no-arbitrage diagnostics
try:
    na_diag = pd.read_csv(NA_URL)
    print(f'No-arb diagnostics from GitHub: {len(na_diag):,} rows')
except Exception:
    import os
    na_diag = pd.read_csv(os.path.join('..', 'Data', 'no_arbitrage_clean_results.csv'))
    print(f'No-arb diagnostics locally: {len(na_diag):,} rows')

na_diag['time_elapsed'] = na_diag['time_elapsed'].astype(int)
print('No-arb columns:', list(na_diag.columns))
na_diag[['max_cond1', 'max_cond2', 'butterfly_ok', 'calendar_ok']].describe().round(4)

## 3. Load market data

### S&P 500 daily prices
Used to compute **realized volatility** targets and HAR-RV features.
Downloaded via `yfinance` (Yahoo Finance).

### VIX index
Downloaded from **FRED** (`VIXCLS` series) via `fredapi` — requires a personal API key
(see `FRED_API_KEY` in the setup cell above; register for free at
https://fred.stlouisfed.org/docs/api/api_key.html).
Used only as an exogenous variable in the ARMAX benchmark (Notebook B) —
*never* as a predictor in the primary SSVI-only models.

In [ ]:
# ── S&P 500 from yfinance + VIX from FRED ────────────────────────────────────
import yfinance as yf

sp500_raw, vix_raw = None, None

try:
    _sp = yf.download('^GSPC', start=MKT_START, end=MKT_END,
                      progress=False, auto_adjust=True)
    if isinstance(_sp.columns, pd.MultiIndex):
        _sp.columns = [c[0].lower() for c in _sp.columns]
    sp500_raw = _sp['close'].squeeze()
    sp500_raw.name = 'sp500'
    sp500_raw.index = pd.to_datetime(sp500_raw.index).normalize()
    print(f'SP500 loaded:  {len(sp500_raw):,} trading days  '
          f'[{sp500_raw.index[0].date()} – {sp500_raw.index[-1].date()}]')
except Exception as e:
    print(f'SP500 download failed: {e}')

if HAS_FREDAPI and FRED_API_KEY and FRED_API_KEY != "PASTE_YOUR_FRED_API_KEY_HERE":
    try:
        fred = Fred(api_key=FRED_API_KEY)
        vix_raw = fred.get_series('VIXCLS', observation_start=MKT_START, observation_end=MKT_END)
        vix_raw.index = pd.to_datetime(vix_raw.index)
        vix_raw = vix_raw.dropna()
        print(f'VIX loaded:    {len(vix_raw):,} trading days  '
              f'[{vix_raw.index[0].date()} – {vix_raw.index[-1].date()}]')
    except Exception as e:
        print(f'VIX (FRED) download failed: {e}')
elif not HAS_FREDAPI:
    print('fredapi not available — VIX will be skipped (pip install fredapi)')
else:
    print('FRED_API_KEY not set — VIX will be skipped. Paste your personal key in the setup '
          'cell above (register for free: https://fred.stlouisfed.org/docs/api/api_key.html)')


## 4. SSVI-derived features

### 4.1 ATM implied volatility

At $k=0$ the SSVI formula collapses to:

$$
\omega(0, T) = \theta_T = e^\alpha T^\beta
$$

This is the **ATM total implied variance** for maturity $T$.
The ATM annualized implied volatility at a reference maturity $T_0$ is:

$$
\sigma_{ATM}(T_0) = \sqrt{\frac{\theta_{T_0}}{T_0}}
= e^{\alpha/2} \cdot T_0^{(\beta-1)/2}
$$

**Portability**: this is entirely a function of $(\alpha, \beta)$ and the chosen
reference maturity — no market price of the underlying is needed after calibration.
Any asset for which SSVI has been calibrated yields a directly comparable ATM IV series.

### 4.2 `max_cond1` — butterfly arbitrage proximity

The **butterfly no-arbitrage condition** requires the local risk-neutral density to be
non-negative everywhere. For SSVI this translates into the constraint:

$$
g(k, T) := \frac{\left(1 - \tfrac{k\phi\rho}{2}\right)^2}{\left(1 + \phi^2(1-\rho^2)\right)}
- \frac{(\phi k + \rho)^2 + 1 - \rho^2}{4}\cdot \phi^2 \geq 0
$$

`max_cond1` $= \max_k g(k,T)^{-1}$ (or a normalized version) captures **how close
the surface is to the butterfly arbitrage boundary**:

- **Low `max_cond1`** → surface has moderate curvature, well inside the no-arb region.
- **`max_cond1` → 0** → surface is at maximum curvature permitted without arbitrage;
  the risk-neutral density $q(S_T)$ is nearly zero at the corresponding strike.
  This is a **local stress indicator**: the market is pricing extreme tail scenarios
  right up to the arbitrage boundary.

**Geometric interpretation**: imagine the SSVI slice as a parabola in $(k, \omega)$ space.
As curvature increases, the parabola becomes tighter; `max_cond1` → 0 means the
tightest permissible shape — after which the density would go negative (illegal).
Historically, `max_cond1` spikes just before realized volatility events (GFC 2009,
Volmageddon 2018, COVID 2020), making it a natural forward-looking stress signal
**derived purely from the options surface**.

### 4.3 Skew stress

$$
\text{skew\_stress}_t = |\rho_t| \cdot \eta_t
$$

The slope of the SSVI surface at $k=0$ is $\frac{\partial\omega}{\partial k}\big|_{k=0} = \theta\phi\rho/2$.
Multiplying $|\rho|$ (direction of skew) by $\eta$ (amplitude of the smile) yields
a scale-free measure of market skewness intensity — high values correspond to
periods when options traders are paying a large premium for downside protection.

In [ ]:
# ── ATM IV at T0 = 1/12 year (≈ 21 trading days / 252) ──────────────────────
T_REF = 21 / 252  # 1-month reference maturity in years

theta_ref = np.exp(ssvi['alpha']) * (T_REF ** ssvi['beta'])
ssvi['atm_iv'] = np.sqrt(theta_ref / T_REF)

# ── No-arb diagnostics: merge into ssvi ──────────────────────────────────────
keep_na = ['time_elapsed', 'max_cond1', 'max_cond2', 'butterfly_ok', 'calendar_ok', 'no_arbitrage']
keep_na = [c for c in keep_na if c in na_diag.columns]
daily = ssvi.merge(na_diag[keep_na], on='time_elapsed', how='left')

# ── SSVI-derived features ─────────────────────────────────────────────────────
daily['abs_rho']          = daily['rho'].abs()
daily['skew_stress']      = daily['abs_rho'] * daily['eta']
daily['eta_gamma']        = daily['eta'] * daily['gamma']

# First differences of SSVI parameters (used in Notebook B)
for p in ['alpha', 'beta', 'rho', 'eta', 'gamma']:
    daily[f'd_{p}'] = daily[p].diff()

print('SSVI-derived features computed:')
for col in ['atm_iv', 'abs_rho', 'skew_stress', 'eta_gamma',
             'd_alpha', 'd_beta', 'd_rho', 'd_eta', 'd_gamma']:
    s = daily[col].dropna()
    print(f'  {col:20s}  mean={s.mean():.4f}  std={s.std():.4f}')

## 5. HAR-RV features

The **Heterogeneous Autoregressive model** (Corsi 2009) decomposes realized volatility
into contributions from different time scales:

$$
\log RV_{t+1} = c + \beta_d \log RV_t^{(d)}
+ \beta_w \log RV_t^{(w)}
+ \beta_m \log RV_t^{(m)}
+ \varepsilon_{t+1}
$$

where:

| Component | Formula | Horizon | Agents captured |
|-----------|---------|---------|----------------|
| Daily $RV_t^{(d)}$ | $\sqrt{252 \cdot r_t^2}$ | 1 day | High-frequency traders |
| Weekly $RV_t^{(w)}$ | $\frac{1}{5}\sum_{i=1}^{5}\sqrt{252\cdot r_{t-i+1}^2}$ | 5 days | Short-term funds |
| Monthly $RV_t^{(m)}$ | $\frac{1}{22}\sum_{i=1}^{22}\sqrt{252\cdot r_{t-i+1}^2}$ | 22 days | Long-term investors |

**Multi-step targets** (Notebook C, horizons $h = 1, 5, 20$):
$$
y_t^{(h)} = \frac{1}{h}\sum_{i=1}^{h}\log RV_{t+i}
= \text{rolling}_h(\log RV).\text{mean}().\text{shift}(-h)
$$

The $\text{shift}(-h)$ aligns the $h$-step-ahead mean with the features at time $t$,
ensuring no look-ahead. The rolling mean is computed *before* shifting.

In [ ]:
if sp500_raw is not None:
    # Align SP500 dates with time_elapsed index
    # The SSVI dataset uses a sequential integer index for trading days
    # We need to map calendar dates to the same integer keys
    sp500_sorted = sp500_raw.sort_index().reset_index(drop=False)
    sp500_sorted.columns = ['date', 'sp500']
    sp500_sorted = sp500_sorted[sp500_sorted['sp500'].notna()].reset_index(drop=True)
    # trading-day sequence index matching SSVI time_elapsed (100 = first day)
    sp500_sorted['te_approx'] = sp500_sorted.index + 100

    # Log returns
    sp500_sorted['log_ret']    = np.log(sp500_sorted['sp500'] / sp500_sorted['sp500'].shift(1))
    sp500_sorted['sq_ret']     = sp500_sorted['log_ret'] ** 2

    # Annualized daily RV
    sp500_sorted['rv_d'] = np.sqrt(252.0 * sp500_sorted['sq_ret'])
    sp500_sorted['log_rv_d'] = np.log(sp500_sorted['rv_d'].clip(lower=1e-8))

    # HAR components (backward-looking — no look-ahead)
    sp500_sorted['rv_w'] = sp500_sorted['rv_d'].rolling(5,  min_periods=5).mean()
    sp500_sorted['rv_m'] = sp500_sorted['rv_d'].rolling(22, min_periods=22).mean()

    print('HAR-RV features computed:')
    for col in ['rv_d', 'rv_w', 'rv_m']:
        s = sp500_sorted[col].dropna()
        print(f'  {col:6s}  n={len(s):,}  mean={s.mean():.4f}  std={s.std():.4f}')

    # Multi-step log-RV targets (h = 1, 5, 20)
    log_rv = sp500_sorted['log_rv_d']
    for h in [1, 5, 20]:
        sp500_sorted[f'log_rv_target_h{h}'] = log_rv.rolling(h).mean().shift(-h)

    print('\nMulti-step targets (forward-looking, aligned via shift):')
    for h in [1, 5, 20]:
        col = f'log_rv_target_h{h}'
        s = sp500_sorted[col].dropna()
        print(f'  h={h:2d}  n={len(s):,}  mean={s.mean():.4f}  std={s.std():.4f}')
else:
    print('SP500 data not available — HAR features skipped.')

## 6. VIX-derived features (exogenous only)

VIX features are constructed for use **exclusively** as exogenous variables
in the ARMAX benchmark (Notebook B). They are **not** included in the primary
SSVI-only forecasting models (Notebook C).

| Feature | Formula | Interpretation |
|---------|---------|---------------|
| `log_vix` | $\ln(\text{VIX}_t)$ | Level of implied volatility regime |
| `d_log_vix` | $\Delta\ln(\text{VIX}_t)$ | Daily shock to volatility expectations |
| `vix_zscore` | $(\ln\text{VIX}_t - \mu_{252}) / \sigma_{252}$ | Rolling standardized level |

**Lagging convention (no look-ahead)**: ARMAX uses `d_log_vix.shift(1)` and
`vix_zscore.shift(1)` — yesterday's VIX shock predicts today's SSVI parameter,
not contemporaneous VIX which would constitute leakage.

In [ ]:
if vix_raw is not None:
    vix = vix_raw.to_frame('vix').reset_index()
    vix.columns = ['date', 'vix']
    vix = vix.dropna().reset_index(drop=True)

    vix['log_vix']    = np.log(vix['vix'])
    vix['d_log_vix']  = vix['log_vix'].diff()

    # 252-day rolling z-score of log-VIX
    roll = vix['log_vix'].rolling(252, min_periods=126)
    vix['vix_zscore'] = (vix['log_vix'] - roll.mean()) / roll.std()

    print('VIX exogenous features computed:')
    for col in ['log_vix', 'd_log_vix', 'vix_zscore']:
        s = vix[col].dropna()
        print(f'  {col:14s}  n={len(s):,}  mean={s.mean():.4f}  std={s.std():.4f}')
else:
    print('VIX data not available — exogenous features skipped.')

## 7. Look-ahead validation

A rigorous check that no feature at time $t$ uses information from $t+1, \ldots$.

**Rules checked**:
1. All backward-rolling windows end at $t$ (min index used ≤ $t$)
2. All lag features are shifted forward in time (constructed via `.shift(k)` with $k \geq 1$)
3. Forward-looking quantities appear **only** in target columns (via `.shift(-h)` applied
   last, after all rolling, so features remain past-only)
4. VIX exogenous variables are shifted by 1 before entering the model

The table below maps each feature family to the data horizon it requires.

In [ ]:
lookahead_audit = pd.DataFrame([
    # (feature_name, uses_data_from, is_safe)
    ('alpha, beta, rho, eta, gamma',  'calibrated at t (close prices t)',     True),
    ('d_alpha, d_beta, ...',          'alpha_t - alpha_{t-1}',                True),
    ('atm_iv',                        'exp(alpha) * T0^beta at t',            True),
    ('max_cond1, max_cond2',          'no-arb diagnostics at t',              True),
    ('skew_stress = |rho|*eta',       'SSVI parameters at t',                 True),
    ('rv_d',                          'squared return at t',                  True),
    ('rv_w (5-day mean)',             'returns t-4 through t',                True),
    ('rv_m (22-day mean)',            'returns t-21 through t',               True),
    ('log_vix (ARMAX exog)',          'VIX close at t, shifted to t+1',       True),
    ('d_log_vix (ARMAX exog)',        'log_vix_t - log_vix_{t-1}, shifted',   True),
    ('log_rv_target_h1',              'log_rv at t+1 (TARGET only)',          True),
    ('log_rv_target_h5',              'mean log_rv at t+1..t+5 (TARGET)',     True),
    ('log_rv_target_h20',             'mean log_rv at t+1..t+20 (TARGET)',    True),
], columns=['Feature', 'Data horizon', 'Safe (no look-ahead)'])

print(lookahead_audit.to_string(index=False))
print(f'\nAll features safe: {lookahead_audit["Safe (no look-ahead)"].all()}')

## 8. SSVI features — exploratory plots

In [ ]:
# SSVI parameter time series
params  = ['alpha', 'beta', 'rho', 'eta', 'gamma']
ylabels = [r'$\alpha$ (log-level)', r'$\beta$ (term-structure slope)',
            r'$\rho$ (skew)', r'$\eta$ (smile amplitude)',
            r'$\gamma$ (curvature decay)']

fig, axes = plt.subplots(3, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle('SSVI parameters — daily time series (2010–2020)', fontsize=11)
te = daily['time_elapsed']

for ax, col, lab in zip(axes.flat[:5], params, ylabels):
    raw  = daily[col]
    smth = raw.rolling(20, center=True).mean()
    ax.plot(te, raw,  lw=0.5, color='steelblue', alpha=0.35)
    ax.plot(te, smth, lw=1.6, color='steelblue', label='20d MA')
    ax.set_ylabel(lab, fontsize=9)
    ax.set_xlabel('Time elapsed (trading days)', fontsize=8)
    ax.legend(fontsize=8)

# ATM IV
ax = axes.flat[5]
ax.plot(te, daily['atm_iv'],                      lw=0.5, color='#E8602B', alpha=0.35)
ax.plot(te, daily['atm_iv'].rolling(20,center=True).mean(), lw=1.6, color='#E8602B', label='20d MA')
ax.set_ylabel(r'$\sigma_{ATM}(T_0)$ — 1M ATM IV', fontsize=9)
ax.set_xlabel('Time elapsed (trading days)', fontsize=8)
ax.legend(fontsize=8)

plt.savefig('S1_ssvi_params_timeseries.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# max_cond1 — butterfly proximity indicator
if 'max_cond1' in daily.columns:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, constrained_layout=True)
    fig.suptitle('max_cond1 — butterfly arbitrage proximity indicator', fontsize=11)

    ax = axes[0]
    ax.plot(te, daily['max_cond1'], lw=0.8, color='#9B59B6', alpha=0.6)
    ax.plot(te, daily['max_cond1'].rolling(20, center=True).mean(),
             lw=1.8, color='#9B59B6', label='20d MA')
    ax.axhline(daily['max_cond1'].quantile(0.90), color='crimson', lw=1, ls='--',
                label='90th pct (stress zone)')
    ax.set_ylabel('max_cond1', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_title('max_cond1 over time — spikes near vol events', fontsize=9)

    ax = axes[1]
    ax.plot(te, daily['atm_iv'], lw=0.8, color='steelblue', alpha=0.6)
    ax.plot(te, daily['atm_iv'].rolling(20, center=True).mean(),
             lw=1.8, color='steelblue', label='20d MA')
    ax.set_ylabel(r'$\sigma_{ATM}$ (1M)', fontsize=9)
    ax.set_xlabel('Time elapsed (trading days)', fontsize=8)
    ax.legend(fontsize=8)
    ax.set_title('ATM IV — for comparison', fontsize=9)

    plt.savefig('S1_maxcond1_vs_atm_iv.png', dpi=110, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation heatmap — SSVI features and derived quantities
hm_cols = [c for c in ['alpha','beta','rho','eta','gamma',
                         'atm_iv','abs_rho','skew_stress','max_cond1','max_cond2',
                         'd_alpha','d_rho']
            if c in daily.columns]

corr_mat = daily[hm_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_mat.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
ax.set_xticks(range(len(hm_cols)))
ax.set_yticks(range(len(hm_cols)))
ax.set_xticklabels(hm_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(hm_cols, fontsize=9)
for i in range(len(hm_cols)):
    for j in range(len(hm_cols)):
        v = corr_mat.values[i, j]
        if abs(v) > 0.25:
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=7, color='white' if abs(v) > 0.6 else 'black')
ax.set_title('Correlation matrix — SSVI features and derived quantities', fontsize=10)
plt.tight_layout()
plt.savefig('S1_ssvi_correlation_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()

## 9. Feature inventory summary

Complete list of all features produced by this pipeline, with their look-ahead
status and the downstream notebooks that consume them.

In [ ]:
feature_inventory = pd.DataFrame([
    # ── SSVI levels ────────────────────────────────────────────────────────────
    ('alpha',        'SSVI',      'Level of ATM variance (log-scale)',              'B, C'),
    ('beta',         'SSVI',      'Term-structure slope of total variance',         'B, C'),
    ('rho',          'SSVI',      'Correlation / skew parameter',                   'B, C'),
    ('eta',          'SSVI',      'Smile amplitude (curvature magnitude)',           'B, C'),
    ('gamma',        'SSVI',      'Curvature decay with ATM variance',              'B, C'),
    # ── SSVI differences ───────────────────────────────────────────────────────
    ('d_alpha',      'SSVI Δ',    '1-day change in alpha',                          'B'),
    ('d_beta',       'SSVI Δ',    '1-day change in beta',                           'B'),
    ('d_rho',        'SSVI Δ',    '1-day change in rho',                            'B'),
    ('d_eta',        'SSVI Δ',    '1-day change in eta',                            'B'),
    ('d_gamma',      'SSVI Δ',    '1-day change in gamma',                          'B'),
    # ── SSVI derived ───────────────────────────────────────────────────────────
    ('atm_iv',       'SSVI der.', '1M ATM IV = sqrt(theta_T0 / T0)',               'B, C'),
    ('abs_rho',      'SSVI der.', '|rho| — unsigned skew intensity',                'B, C'),
    ('skew_stress',  'SSVI der.', '|rho|*eta — scale-free skewness measure',        'B, C'),
    ('eta_gamma',    'SSVI der.', 'eta*gamma — interaction',                        'C'),
    ('max_cond1',    'No-arb',    'Butterfly arbitrage proximity (stress signal)',   'B, C'),
    ('max_cond2',    'No-arb',    'Calendar spread arbitrage margin',               'B'),
    # ── HAR-RV ─────────────────────────────────────────────────────────────────
    ('rv_d',         'HAR-RV',    'Daily realized vol (annualized)',                'C'),
    ('rv_w',         'HAR-RV',    '5-day mean RV (weekly component)',               'C'),
    ('rv_m',         'HAR-RV',    '22-day mean RV (monthly component)',             'C'),
    ('log_rv_d',     'HAR-RV',    'log(rv_d) — model target space',                'C'),
    # ── VIX exogenous ──────────────────────────────────────────────────────────
    ('log_vix',      'VIX (FRED)','log(VIX) — exogenous regime proxy',             'B (ARMAX)'),
    ('d_log_vix',    'VIX (FRED)','Δlog(VIX) — daily VIX shock (lagged)',          'B (ARMAX)'),
    ('vix_zscore',   'VIX (FRED)','Rolling z-score of log-VIX (lagged)',           'B (ARMAX)'),
    # ── Targets ────────────────────────────────────────────────────────────────
    ('log_rv_target_h1',  'TARGET', 'Mean log-RV over next 1 day',                 'C (h=1)'),
    ('log_rv_target_h5',  'TARGET', 'Mean log-RV over next 5 days',                'C (h=5)'),
    ('log_rv_target_h20', 'TARGET', 'Mean log-RV over next 20 days',               'C (h=20)'),
], columns=['Feature', 'Family', 'Description', 'Used in'])

pd.set_option('display.max_colwidth', 60)
print(feature_inventory.to_string(index=False))
print(f'\nTotal features: {len(feature_inventory[feature_inventory["Family"] != "TARGET"])}')
print(f'Targets:        {len(feature_inventory[feature_inventory["Family"] == "TARGET"])}')

## 10. Design principles

### Why SSVI parameters as forecasting features?

The risk-neutral measure prices the market's expectation of future variance:
$\mathbb{E}^Q[\text{Var}_t] \approx \omega(0, T)/T = \sigma_{ATM}^2(T)$.
Empirically (Carr & Wu 2009; Bollerslev et al. 2014), the **slope** ($\rho$)
and **curvature** ($\eta, \gamma$) of the implied surface contain forward-looking
information about realized variance *beyond* what backward-looking RV measures
can capture — driven by the variance risk premium and crash-fear dynamics.

### Why not use VIX in primary models?

VIX is a model-free ATM-IV proxy computed from SPX options — **a transformation
of the same options data** used to calibrate SSVI. Including VIX in an SSVI-feature
model creates redundancy, not orthogonal information. More importantly, portability
requires that models work wherever SSVI can be calibrated: equity indices, FX rates,
commodity options — none of which have a VIX equivalent. SSVI parameters are
portable; VIX is not.

### Temporal train/test split

A random split would allow the model to interpolate between training observations
straddling a test point, leaking future information through autocorrelation.
The **temporal split** (train = first 80% of calendar time, test = last 20%)
simulates real deployment: the model is trained on history and evaluated on the future.

### MAPE vs RMSE

Realized volatility spans several orders of magnitude across regimes
(0.05 in quiet periods, >1.0 during COVID). RMSE overweights crisis periods;
**MAPE** (mean absolute percentage error) gives equal relative weight across regimes,
making it the natural complement to RMSE for assessing model calibration in
both calm and stressed markets.